# **Judulmu: Deteksi Objek Helm pada Pengendara Sepeda Motor Menggunakan Metode YOLO26 sebagai Upaya Peningkatan Keselamatan Berlalu Lintas.**

**LANGKAH 1: Cek GPU**

In [ ]:
# nvidia-smi = menampilkan info kartu grafis (GPU) yang sedang dipakai.
!nvidia-smi

**LANGKAH 2: Install library, sambungkan Drive, atur path**

In [ ]:
# Install library YOLO dan Roboflow
!pip install -q ultralytics roboflow

In [ ]:
import os
from google.colab import drive

# Sambungkan Google Drive (klik Allow pada popup)
drive.mount('/content/drive')

In [ ]:
# Lokasi folder proyek di Drive
PROJECT_DIR = "/content/drive/MyDrive/ProjekUTS_Kelompok1PCD"
DATASET_DIR = f"{PROJECT_DIR}/datasets"   # Dataset
RESULT_DIR  = f"{PROJECT_DIR}/hasil"      # Hasil training

In [ ]:
# Pengaturan yang dipakai di langkah berikutnya
DATA_YAML = f"{DATASET_DIR}/data_helm.yaml"            # Konfigurasi dataset 3 kelas
NAMA_BARU = ["helmet", "no-helmet", "motorcyclist"]    # 0 = helmet, 1 = no-helmet, 2 = motorcyclist
NAMA_RUN  = "helm_3kelas"                              # Nama folder percobaan
RUN       = f"{RESULT_DIR}/{NAMA_RUN}"                 # Folder hasil training ini

In [ ]:
# Cek folder proyek dan tampilkan isinya
print("Folder proyek ada:", os.path.exists(PROJECT_DIR))
if os.path.exists(PROJECT_DIR):
    os.makedirs(RESULT_DIR, exist_ok=True)    # Buat folder hasil jika belum ada
    print("Isi folder:", os.listdir(PROJECT_DIR))
print("Path siap.")

**LANGKAH 3: Cek dan download dataset**

In [ ]:
# Download dataset format YOLO26 langsung ke Drive
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="lSw0vEBAWd11Uq3ktKaD")
project = rf.workspace("helmet-and-number-plate-detection-project").project("helmet-and-number-plate-detection-for-motorbike-safety-iityz")
version = project.version(3)

dataset = version.download("yolo26", location=DATASET_DIR)

In [ ]:
# Hitung jumlah gambar di folder train, valid, dan test
for s in ["train", "valid", "test"]:
    p = f"{DATASET_DIR}/{s}/images"
    print(s, len(os.listdir(p)) if os.path.exists(p) else "TIDAK ADA")

# Tampilkan isi folder dataset
print("Isi folder:", os.listdir(DATASET_DIR))

In [ ]:
# Library untuk membaca file .yaml (file konfigurasi dataset)
import yaml

# Buka file data.yaml yang dibuat Roboflow, isinya info kelas dataset
with open(f"{DATASET_DIR}/data.yaml") as f:
    cfg = yaml.safe_load(f)   # Ubah isi file jadi dictionary Python

# nc = number of classes (jumlah kelas), names = daftar nama kelas
print("Jumlah kelas:", cfg["nc"])
print("Nama kelas  :", cfg["names"])

# Tampilkan 1500 karakter pertama README untuk melihat deskripsi dataset
print("\n--- README.dataset.txt ---")
print(open(f"{DATASET_DIR}/README.dataset.txt").read()[:1500])

In [ ]:
# glob = mencari file berdasarkan pola nama, random = mengambil data acak
import glob, random

# Counter = alat bantu untuk menghitung berapa kali sesuatu muncul
from collections import Counter

# Ambil daftar semua file label (.txt) di folder train
files = glob.glob(f"{DATASET_DIR}/train/labels/*.txt")

# Ambil 2000 file secara acak saja supaya tidak lama membaca dari Drive
sampel = random.sample(files, min(2000, len(files)))

# Wadah untuk menghitung jumlah objek di tiap kelas
hitung = Counter()

# Baca satu per satu file label dari sampel tadi
for txt in sampel:
    for baris in open(txt):          # Satu baris = satu objek di gambar
        if baris.strip():            # Lewati baris kosong
            # Angka pertama di tiap baris adalah nomor kelas (0, 1, 2, ...)
            hitung[int(baris.split()[0])] += 1

# Tampilkan hasil hitungan, diurutkan dari nomor kelas terkecil
print(f"Distribusi objek (sampel {len(sampel)} file label):")
for k in sorted(hitung):
    print(f"  kelas {k} ({cfg['names'][k]}): {hitung[k]}")

**Tampilkan contoh gambar dengan semua kelas asli.**

In [ ]:
# cv2 = opencv untuk baca dan gambar kotak, matplotlib = untuk menampilkan gambar
import cv2, glob, random
import matplotlib.pyplot as plt

# Nama kelas asli dataset (urutan sesuai data.yaml)
NAMA_ASLI = ["bike", "helmet", "no-helmet", "number-plate"]

# Warna kotak (RGB): bike kuning, helmet hijau, no-helmet merah, number-plate biru
WARNA_ASLI = [(255, 180, 0), (0, 200, 0), (220, 0, 0), (0, 120, 255)]

def gambar_kotak(path_gambar):
    # Baca gambar lalu ubah BGR ke RGB supaya warna benar
    img = cv2.cvtColor(cv2.imread(path_gambar), cv2.COLOR_BGR2RGB)
    tinggi, lebar = img.shape[:2]

    # File label punya nama sama dengan gambar, ekstensinya .txt
    nama = os.path.splitext(os.path.basename(path_gambar))[0]
    path_label = f"{DATASET_DIR}/train/labels/{nama}.txt"
    kelas_ada = set()

    if os.path.exists(path_label):
        for baris in open(path_label):
            b = baris.split()
            if len(b) < 5:
                continue
            k = int(b[0])
            kelas_ada.add(NAMA_ASLI[k])
            xc, yc, w, h = map(float, b[1:5])
            # Ubah skala 0-1 menjadi piksel
            x1, y1 = int((xc - w / 2) * lebar), int((yc - h / 2) * tinggi)
            x2, y2 = int((xc + w / 2) * lebar), int((yc + h / 2) * tinggi)
            cv2.rectangle(img, (x1, y1), (x2, y2), WARNA_ASLI[k], 3)
            cv2.putText(img, NAMA_ASLI[k], (x1, max(y1 - 6, 14)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, WARNA_ASLI[k], 2)
    return img, kelas_ada

# Ambil 12 gambar acak dari folder train
semua_gambar = glob.glob(f"{DATASET_DIR}/train/images/*")
contoh = random.sample(semua_gambar, 12)

# Kanvas 3 baris x 4 kolom
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
for ax, p in zip(axes.ravel(), contoh):
    img, kelas_ada = gambar_kotak(p)
    ax.imshow(img)
    ax.axis("off")
    # Judul = daftar kelas yang ada di gambar itu
    ax.set_title(", ".join(sorted(kelas_ada)) or "TANPA LABEL", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Library untuk menyalin folder
import shutil

for s in ["train", "valid", "test"]:
    asli = f"{DATASET_DIR}/{s}/labels_asli"
    if os.path.exists(asli):
        print(s, "cadangan sudah ada, dilewati")   # Jangan menimpa cadangan
    else:
        shutil.copytree(f"{DATASET_DIR}/{s}/labels", asli)   # Salin seluruh folder labels
        print(s, "cadangan dibuat")

**LANGKAH 4: Buat dataset 3 kelas**

In [ ]:
import os, glob

PETA_3 = {1: 0, 2: 1}       # helmet -> 0, no-helmet -> 1
ID_MOTORCYCLE_ASLI = 0      # Di dataset asli, kelas bike (motor beserta pengendara)
ID_PENGENDARA = 2           # Kelas baru: pengendara-motor

def ubah_split_3(split):
    src = f"{DATASET_DIR}/{split}/labels"          # Sumber: label asli (tidak diubah)
    tmp = f"{DATASET_DIR}/{split}/labels_3kelas"   # Hasil ditulis ke sini dulu
    lama = f"{DATASET_DIR}/{split}/labels_lama"    # Tempat label asli setelah ditukar

    if os.path.exists(lama):
        print(split, "sudah dikonversi, dilewati")
        return
    os.makedirs(tmp, exist_ok=True)

    n_file, n_poligon, n_rider = 0, 0, 0
    for f in sorted(glob.glob(f"{src}/*.txt")):
        kepala, motorcycle = [], []
        for baris in open(f):
            p = baris.split()
            if len(p) < 5:
                continue                              # Lewati baris kosong atau rusak
            kelas = int(float(p[0]))
            if kelas not in PETA_3 and kelas != ID_MOTORCYCLE_ASLI:
                continue                              # Buang number-plate

            if len(p) > 5:                            # Baris poligon: ambil kotak pembungkus
                xs = [float(v) for v in p[1::2]]
                ys = [float(v) for v in p[2::2]]
                x1, x2 = max(0, min(xs)), min(1, max(xs))
                y1, y2 = max(0, min(ys)), min(1, max(ys))
                n_poligon += 1
            else:                                     # Baris kotak biasa
                cx, cy, w, h = [float(v) for v in p[1:5]]
                x1, x2, y1, y2 = cx - w / 2, cx + w / 2, cy - h / 2, cy + h / 2

            (motorcycle if kelas == ID_MOTORCYCLE_ASLI else kepala).append((kelas, x1, y1, x2, y2))

        hasil = []
        for kelas, x1, y1, x2, y2 in kepala:          # Kelas helmet dan no-helmet
            hasil.append(f"{PETA_3[kelas]} {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}")

        # Kelas baru: gabung kotak motor dengan kepala di atasnya
        gabung = {}
        for _, hx1, hy1, hx2, hy2 in kepala:
            hcx, hcy = (hx1 + hx2) / 2, (hy1 + hy2) / 2
            calon = []
            for i, (_, mx1, my1, mx2, my2) in enumerate(motorcycle):
                if mx1 <= hcx <= mx2 and my1 - 1.5 * (my2 - my1) <= hcy <= my2:
                    calon.append((abs(hcx - (mx1 + mx2) / 2), i))
            if not calon:
                continue                              # Kepala tanpa motor
            i = min(calon)[1]
            _, mx1, my1, mx2, my2 = motorcycle[i]
            x1, y1, x2, y2 = gabung.get(i, (mx1, my1, mx2, my2))
            gabung[i] = (min(x1, hx1), min(y1, hy1), max(x2, hx2), max(y2, hy2))

        for x1, y1, x2, y2 in gabung.values():
            hasil.append(f"{ID_PENGENDARA} {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}")
            n_rider += 1

        # Tulis ke folder sementara, file kosong berarti gambar tanpa objek
        with open(f"{tmp}/{os.path.basename(f)}", "w") as out:
            out.write("\n".join(hasil) + ("\n" if hasil else ""))
        n_file += 1

    # Tukar folder: label asli -> labels_lama, hasil baru -> labels
    os.rename(src, lama)
    os.rename(tmp, src)

    # Hapus cache lama supaya YOLO membaca label yang baru
    cache = f"{DATASET_DIR}/{split}/labels.cache"
    if os.path.exists(cache):
        os.remove(cache)

    print(f"{split}: {n_file} file, {n_poligon} poligon jadi kotak, {n_rider} pengendara motor dibuat")

for s in ["test", "valid", "train"]:
    ubah_split_3(s)

In [ ]:
# Urutan nama harus sama dengan nomor kelas baru (0, 1, 2)
NAMA_BARU = ["helmet", "no-helmet", "pengendara-motor"]

isi_yaml = f"""path: {DATASET_DIR}
train: train/images
val: valid/images
test: test/images
nc: {len(NAMA_BARU)}
names: {NAMA_BARU}
"""

DATA_YAML = f"{DATASET_DIR}/data_helm.yaml"
open(DATA_YAML, "w").write(isi_yaml)
print(isi_yaml)

In [ ]:
import glob
from collections import Counter

# Hitung objek per kelas di label baru, harus hanya ada kelas 0, 1, 2
hitung = Counter()
for txt in glob.glob(f"{DATASET_DIR}/valid/labels/*.txt"):
    for baris in open(txt):
        if baris.strip():
            hitung[int(baris.split()[0])] += 1

for k in sorted(hitung):
    print(f"kelas {k} ({NAMA_BARU[k]}): {hitung[k]}")

In [ ]:
# Mengimpor library os
import os

# Menentukan lokasi dataset
DATASET_PATH = "/content/drive/MyDrive/ProjekUTS_Kelompok1PCD"

# Menampilkan struktur folder dataset
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, "").count(os.sep)

    # Membatasi tampilan struktur hingga 3 tingkat
    if level > 3:
        dirs[:] = []
        continue

    print("  " * level + os.path.basename(root) + "/")

    for file in files[:5]:
        print("  " * (level + 1) + file)

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

def lihat_contoh(split="train", folder_label="labels",
                 nama_kelas=None, seed=1):

    random.seed(seed)

    # Lokasi dataset
    DATASET_PATH = "/content/drive/MyDrive/ProjekUTS_Kelompok1PCD/datasets"

    folder_gambar = os.path.join(DATASET_PATH, split, "images")
    folder_labels = os.path.join(DATASET_PATH, split, folder_label)

    # Nama kelas
    if nama_kelas is None:
        nama_kelas = {
            0: "helmet",
            1: "no-helmet",
            2: "pengendara"
        }

    # Warna bounding box
    WARNA = {
        0: (0, 200, 0),
        1: (255, 0, 0),
        2: (0, 120, 255)
    }

    # Memeriksa folder dataset
    if not os.path.exists(folder_gambar):
        print("Folder gambar tidak ditemukan:", folder_gambar)
        return

    if not os.path.exists(folder_labels):
        print("Folder label tidak ditemukan:", folder_labels)
        return

    gambar_list = [
        f for f in os.listdir(folder_gambar)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if not gambar_list:
        print("Gambar tidak ditemukan!")
        return

    # Memilih maksimal 5 gambar
    sampel = random.sample(gambar_list, min(5, len(gambar_list)))

    fig, axes = plt.subplots(1, len(sampel), figsize=(20, 6))

    if len(sampel) == 1:
        axes = [axes]

    for ax, gambar in zip(axes, sampel):

        img_path = os.path.join(folder_gambar, gambar)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        tinggi, lebar, _ = img.shape

        nama_file = os.path.splitext(gambar)[0]
        label_file = os.path.join(folder_labels, nama_file + ".txt")

        if os.path.exists(label_file):
            with open(label_file, "r") as f:
                labels = f.readlines()

            for baris in labels:
                data = baris.strip().split()

                if len(data) != 5:
                    continue

                id_kelas = int(data[0])
                x_center, y_center, w, h = map(float, data[1:])

                # Mengubah koordinat YOLO menjadi piksel
                x_center *= lebar
                y_center *= tinggi
                w *= lebar
                h *= tinggi

                x1 = int(x_center - w / 2)
                y1 = int(y_center - h / 2)
                x2 = int(x_center + w / 2)
                y2 = int(y_center + h / 2)

                warna = WARNA.get(id_kelas, (255, 255, 255))
                nama = nama_kelas.get(id_kelas, str(id_kelas))

                cv2.rectangle(img, (x1, y1), (x2, y2), warna, 2)
                cv2.putText(
                    img, nama, (x1, max(y1 - 10, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, warna, 2
                )

        ax.imshow(img)
        ax.set_title(gambar, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Menampilkan 5 contoh gambar training beserta bounding box dan nama kelas
lihat_contoh(
    split="train",
    folder_label="labels",
    nama_kelas=NAMA_BARU,
    seed=4
)

In [ ]:
# Menampilkan 5 contoh gambar testing beserta bounding box dan nama kelas
lihat_contoh(
    split="test",
    folder_label="labels",
    nama_kelas=NAMA_BARU,
    seed=13
)

In [ ]:
# Menampilkan daftar nama kelas berdasarkan ID
print("Daftar kelas:")

for id_kelas, nama in NAMA_BARU.items():
    print(f"{id_kelas}: {nama}")

**LANGKAH 5: Pisahkan gambar pakai COCO**

In [ ]:
# YOLO COCO = model bawaan yang sudah mengenali orang, sepeda, dan motor
from ultralytics import YOLO
coco = YOLO("yolo11m.pt")
ID_PERSON, ID_BICYCLE, ID_MOTOR = 0, 1, 3

def luas(b):
    # Luas kotak format (x1, y1, x2, y2)
    return max(0, b[2] - b[0]) * max(0, b[3] - b[1])

def irisan(a, b):
    # Luas daerah tumpang tindih antara dua kotak
    return luas((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))

def baca_kepala(path_label):
    # Ambil kotak kepala (kelas 0 helmet dan 1 no-helmet) dari file label
    hasil = []
    for b in open(path_label).read().split("\n"):
        q = b.split()
        if len(q) >= 5 and q[0] in ("0", "1"):
            cx, cy, w, h = map(float, q[1:5])
            hasil.append((cx - w/2, cy - h/2, cx + w/2, cy + h/2))
    return hasil

def cari_pengendara(orang, motor, kepala):
    # Pengendara = motor + orang yang duduk di atasnya (ada kepala berlabel di dalam kotak orang)
    hasil = []
    for m in motor:
        penumpang = []
        for p in orang:
            if luas(p) == 0:
                continue
            tengah_x = (p[0] + p[2]) / 2
            dalam_lebar = m[0] <= tengah_x <= m[2]            # Orang sejajar dengan motor
            menempel = irisan(p, m) / luas(p) >= 0.3          # Badan orang menempel di motor
            ada_kepala = any(p[0] <= (k[0]+k[2])/2 <= p[2] and
                             p[1] <= (k[1]+k[3])/2 <= p[3] for k in kepala)
            if dalam_lebar and menempel and ada_kepala:
                penumpang.append(p)
        if penumpang:
            semua = penumpang + [m]
            # Gabungkan motor dan penumpang menjadi satu kotak besar
            hasil.append((min(k[0] for k in semua), min(k[1] for k in semua),
                          max(k[2] for k in semua), max(k[3] for k in semua)))
    return hasil

# selamat[split][path_gambar] = kotak pengendara baru; dibuang_nomotor[split] = daftar gambar tanpa motor
selamat, dibuang_nomotor = {}, {}
for s in ["train", "valid", "test"]:
    selamat[s], dibuang_nomotor[s] = {}, []
    daftar = kepala_saja[s]                                    # Dibuat oleh sel penghitung sebelumnya
    for i in range(0, len(daftar), 32):                        # Proses 32 gambar sekali jalan
        grup = daftar[i:i+32]
        hasil = coco.predict(grup, classes=[ID_PERSON, ID_BICYCLE, ID_MOTOR],
                             conf=0.3, imgsz=640, verbose=False)
        for p, r in zip(grup, hasil):
            nama = os.path.splitext(os.path.basename(p))[0]
            kotak = r.boxes.xyxyn.cpu().numpy()
            kelas = r.boxes.cls.cpu().numpy().astype(int)
            orang = [tuple(k) for k, c in zip(kotak, kelas) if c == ID_PERSON]
            motor = [tuple(k) for k, c in zip(kotak, kelas) if c == ID_MOTOR]
            baru = cari_pengendara(orang, motor, baca_kepala(f"{DATASET_DIR}/{s}/labels/{nama}.txt"))
            if baru:
                selamat[s][p] = baru                           # Ada motor + pengendara: diselamatkan
            else:
                dibuang_nomotor[s].append(p)                   # Tidak ada motor: akan dipindah
    print(f"{s}: diselamatkan={len(selamat[s])}, akan dipindah={len(dibuang_nomotor[s])}")

In [ ]:
import random, cv2
import matplotlib.pyplot as plt

# Atas: contoh yang diselamatkan (kotak biru baru). Bawah: contoh yang akan dipindah
pilih_a = random.sample(list(selamat["train"].items()), min(4, len(selamat["train"])))
pilih_b = random.sample(dibuang_nomotor["train"], min(8, len(dibuang_nomotor["train"])))

fig, axs = plt.subplots(3, 4, figsize=(20, 13))
for ax, (p, kotak_baru) in zip(axs[0], pilih_a):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for x1, y1, x2, y2 in kotak_baru:
        cv2.rectangle(img, (int(x1*w), int(y1*h)), (int(x2*w), int(y2*h)), (0, 120, 255), 4)
    ax.imshow(img); ax.axis("off"); ax.set_title("DISELAMATKAN", fontsize=10)
for ax, p in zip(axs[1:].flat, pilih_b):
    ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title("AKAN DIPINDAH", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Mengambil gambar yang tidak memiliki anotasi
kosong_total = {}

for s in ["train", "valid", "test"]:
    kosong_total[s] = []

    folder_gambar = f"{DATASET_DIR}/{s}/images"
    folder_label = f"{DATASET_DIR}/{s}/labels"

    for gambar in os.listdir(folder_gambar):
        if not gambar.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        nama = os.path.splitext(gambar)[0]
        path_label = f"{folder_label}/{nama}.txt"

        if os.path.exists(path_label):
            with open(path_label, "r") as f:
                isi = f.read().strip()

            if not isi:
                kosong_total[s].append(
                    f"{folder_gambar}/{gambar}"
                )

    print(f"{s}: gambar tanpa anotasi={len(kosong_total[s])}")

In [ ]:
import os, shutil

# Folder tampung di Drive (gambar yang dipindah bisa dikembalikan dari sini)
TAMPUNG = f"{PROJECT_DIR}/dibuang"

# True = hanya menghitung. Ubah ke False hanya setelah hasil Sel 2 kamu cek dan bagus
DRY_RUN = True

for s in ["train", "valid", "test"]:
    # BAGIAN 1: Tulis kotak pengendara-motor (kelas 2) ke label gambar yang diselamatkan
    n_tulis = 0
    for p, kotak_baru in selamat[s].items():
        nama = os.path.splitext(os.path.basename(p))[0]
        f = f"{DATASET_DIR}/{s}/labels/{nama}.txt"
        isi = open(f).read()

        # Lewati kalau sudah ada kelas 2, supaya sel ini aman dijalankan ulang
        if any(b.startswith("2 ") for b in isi.split("\n")):
            continue

        baris = [f"2 {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}"
                 for x1, y1, x2, y2 in kotak_baru]
        if not DRY_RUN:
            with open(f, "w") as out:
                out.write(isi.rstrip("\n") + ("\n" if isi.strip() else "") + "\n".join(baris) + "\n")
        n_tulis += 1

    # BAGIAN 2: Pindahkan gambar tanpa motor, ditambah gambar tanpa objek sama sekali
    daftar_pindah = dibuang_nomotor[s] + kosong_total[s]
    n_pindah = 0
    if not DRY_RUN:
        os.makedirs(f"{TAMPUNG}/tanpa_motor/{s}/images", exist_ok=True)
        os.makedirs(f"{TAMPUNG}/tanpa_motor/{s}/labels", exist_ok=True)
    for p in daftar_pindah:
        if not os.path.exists(p):
            continue                                   # Sudah pernah dipindah
        nama = os.path.splitext(os.path.basename(p))[0]
        if not DRY_RUN:
            shutil.move(p, f"{TAMPUNG}/tanpa_motor/{s}/images/{os.path.basename(p)}")
            lbl = f"{DATASET_DIR}/{s}/labels/{nama}.txt"
            if os.path.exists(lbl):
                shutil.move(lbl, f"{TAMPUNG}/tanpa_motor/{s}/labels/{nama}.txt")
        n_pindah += 1

    # Hapus cache lama supaya YOLO membaca ulang daftar gambar dan label
    cache = f"{DATASET_DIR}/{s}/labels.cache"
    if not DRY_RUN and os.path.exists(cache):
        os.remove(cache)

    print(f"{s}: kotak biru ditulis={n_tulis}, gambar dipindah={n_pindah}")

print("MODE HITUNG SAJA (DRY_RUN)" if DRY_RUN else "SELESAI DITERAPKAN")

In [ ]:
from collections import Counter

# Jumlah gambar dan objek per kelas setelah pembersihan
for s in ["train", "valid", "test"]:
    n_img = len(os.listdir(f"{DATASET_DIR}/{s}/images"))
    hitung = Counter()
    for f in glob.glob(f"{DATASET_DIR}/{s}/labels/*.txt"):
        for b in open(f):
            if b.strip():
                hitung[int(b.split()[0])] += 1
    print(f"{s}: {n_img} gambar, " + ", ".join(f"{NAMA_BARU[k]}={v}" for k, v in sorted(hitung.items())))

In [ ]:
# random = memilih contoh acak, cv2 = baca gambar dan gambar kotak, matplotlib = tampilkan di notebook
import random, cv2
import matplotlib.pyplot as plt

# Baris 1: 4 contoh yang DISELAMATKAN (kotak biru baru dari COCO)
# Baris 2-3: 8 contoh yang AKAN DIPINDAH (dianggap tanpa motor)
pilih_a = random.sample(list(selamat["train"].items()), min(4, len(selamat["train"])))
pilih_b = random.sample(dibuang_nomotor["train"], min(8, len(dibuang_nomotor["train"])))

# Kanvas 3 baris x 4 kolom
fig, axs = plt.subplots(3, 4, figsize=(20, 13))

# Baris atas: gambar yang diselamatkan, lengkap dengan kotak biru pengendara-motor
for ax, (p, kotak_baru) in zip(axs[0], pilih_a):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)   # BGR -> RGB supaya warna benar
    h, w = img.shape[:2]
    for x1, y1, x2, y2 in kotak_baru:
        # Ubah skala 0-1 menjadi piksel, lalu gambar kotak biru
        cv2.rectangle(img, (int(x1*w), int(y1*h)), (int(x2*w), int(y2*h)), (0, 120, 255), 4)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("DISELAMATKAN", fontsize=10)

# Dua baris bawah: gambar yang akan dipindah (tanpa kotak, hanya untuk dinilai)
for ax, p in zip(axs[1:].flat, pilih_b):
    ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    ax.axis("off")
    ax.set_title("AKAN DIPINDAH", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
DRY_RUN = False   # Sebelumnya True. Sekarang benar-benar menulis label dan memindahkan gambar

In [ ]:
import os, shutil

# Folder tampung di Drive (gambar yang dipindah bisa dikembalikan dari sini)
TAMPUNG = f"{PROJECT_DIR}/dibuang"

# False = benar-benar menulis label dan memindahkan gambar
DRY_RUN = False

for s in ["train", "valid", "test"]:
    # BAGIAN 1: Tulis kotak pengendara-motor (kelas 2) ke label gambar yang diselamatkan
    n_tulis = 0
    for p, kotak_baru in selamat[s].items():
        nama = os.path.splitext(os.path.basename(p))[0]
        f = f"{DATASET_DIR}/{s}/labels/{nama}.txt"
        isi = open(f).read()

        # Lewati kalau sudah ada kelas 2
        if any(b.startswith("2 ") for b in isi.split("\n")):
            continue

        baris = [
            f"2 {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}"
            for x1, y1, x2, y2 in kotak_baru
        ]

        if not DRY_RUN:
            with open(f, "w") as out:
                out.write(
                    isi.rstrip("\n")
                    + ("\n" if isi.strip() else "")
                    + "\n".join(baris)
                    + "\n"
                )
        n_tulis += 1

    # BAGIAN 2: Pindahkan gambar tanpa motor dan gambar tanpa objek
    daftar_pindah = dibuang_nomotor[s] + kosong_total[s]
    n_pindah = 0

    if not DRY_RUN:
        os.makedirs(f"{TAMPUNG}/tanpa_motor/{s}/images", exist_ok=True)
        os.makedirs(f"{TAMPUNG}/tanpa_motor/{s}/labels", exist_ok=True)

    for p in daftar_pindah:
        if not os.path.exists(p):
            continue

        nama = os.path.splitext(os.path.basename(p))[0]

        if not DRY_RUN:
            shutil.move(
                p,
                f"{TAMPUNG}/tanpa_motor/{s}/images/{os.path.basename(p)}"
            )

            lbl = f"{DATASET_DIR}/{s}/labels/{nama}.txt"
            if os.path.exists(lbl):
                shutil.move(
                    lbl,
                    f"{TAMPUNG}/tanpa_motor/{s}/labels/{nama}.txt"
                )

        n_pindah += 1

    # Hapus cache lama supaya YOLO membaca ulang daftar gambar dan label
    cache = f"{DATASET_DIR}/{s}/labels.cache"
    if not DRY_RUN and os.path.exists(cache):
        os.remove(cache)

    print(f"{s}: kotak biru ditulis={n_tulis}, gambar dipindah={n_pindah}")

print("MODE HITUNG SAJA (DRY_RUN)" if DRY_RUN else "SELESAI DITERAPKAN")

In [ ]:
import os
import glob
import random
import cv2
import matplotlib.pyplot as plt

# Nama kelas sesuai ID anotasi
NAMA_KELAS = ["helmet", "no-helmet", "pengendara-motor"]

# Jumlah gambar yang ditampilkan per dataset
JUMLAH_GAMBAR = 5

# Menampilkan sampel gambar dari setiap dataset
for s in ["train", "valid", "test"]:
    folder = f"{DATASET_DIR}/{s}/images"
    daftar_gambar = glob.glob(f"{folder}/*")

    # Lewati dataset jika tidak memiliki gambar
    if not daftar_gambar:
        print(f"Tidak ada gambar pada dataset {s}")
        continue

    # Mengambil gambar secara acak
    sampel = random.sample(
        daftar_gambar,
        min(JUMLAH_GAMBAR, len(daftar_gambar))
    )

    # Menyiapkan tampilan gambar
    fig, axes = plt.subplots(
        1, len(sampel),
        figsize=(5 * len(sampel), 5)
    )

    if len(sampel) == 1:
        axes = [axes]

    # Membaca gambar dan menampilkan anotasinya
    for ax, path in zip(axes, sampel):
        gambar = cv2.imread(path)
        gambar = cv2.cvtColor(gambar, cv2.COLOR_BGR2RGB)

        tinggi, lebar = gambar.shape[:2]

        # Membaca file label sesuai nama gambar
        nama = os.path.splitext(os.path.basename(path))[0]
        label_path = f"{DATASET_DIR}/{s}/labels/{nama}.txt"

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for baris in f:
                    data = baris.split()

                    if len(data) < 5:
                        continue

                    # Mengambil ID kelas dan koordinat bounding box
                    kelas = int(data[0])
                    x, y, w, h = map(float, data[1:5])

                    # Mengubah koordinat YOLO menjadi koordinat piksel
                    x1 = int((x - w / 2) * lebar)
                    y1 = int((y - h / 2) * tinggi)
                    x2 = int((x + w / 2) * lebar)
                    y2 = int((y + h / 2) * tinggi)

                    # Menggambar bounding box
                    cv2.rectangle(
                        gambar, (x1, y1), (x2, y2),
                        (0, 255, 0), 2
                    )

                    # Menampilkan nama kelas pada bounding box
                    if kelas < len(NAMA_KELAS):
                        nama_kelas = NAMA_KELAS[kelas]
                        cv2.putText(
                            gambar, nama_kelas,
                            (x1, max(y1 - 5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (255, 0, 0), 2
                        )

        ax.imshow(gambar)
        ax.set_title(os.path.basename(path), fontsize=8)
        ax.axis("off")

    # Menampilkan hasil visualisasi dataset
    plt.suptitle(f"Dataset {s.upper()}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import glob
import random
import cv2
import matplotlib.pyplot as plt

# Nama kelas dan warna bounding box
NAMA_KELAS = ["helmet", "no-helmet", "pengendara-motor"]

WARNA = [
    (0, 255, 0),     # Helmet
    (255, 0, 0),     # No-helmet
    (0, 165, 255)    # Pengendara-motor
]

JUMLAH_GAMBAR = 5

# Menampilkan gambar dari dataset train, valid, dan test
for s in ["train", "valid", "test"]:

    folder = f"{DATASET_DIR}/{s}/images"
    daftar_gambar = glob.glob(f"{folder}/*")

    # Melewati dataset jika tidak memiliki gambar
    if not daftar_gambar:
        print(f"Dataset {s} kosong")
        continue

    # Mengambil gambar secara acak
    sampel = random.sample(
        daftar_gambar,
        min(JUMLAH_GAMBAR, len(daftar_gambar))
    )

    # Menyiapkan tampilan gambar
    fig, axes = plt.subplots(
        1, len(sampel),
        figsize=(6 * len(sampel), 6)
    )

    if len(sampel) == 1:
        axes = [axes]

    # Membaca gambar dan menampilkan anotasinya
    for ax, path in zip(axes, sampel):

        gambar = cv2.imread(path)
        gambar = cv2.cvtColor(gambar, cv2.COLOR_BGR2RGB)

        tinggi, lebar = gambar.shape[:2]

        # Menentukan file label berdasarkan nama gambar
        nama = os.path.splitext(os.path.basename(path))[0]
        label_path = f"{DATASET_DIR}/{s}/labels/{nama}.txt"

        # Membaca koordinat label YOLO
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for baris in f:

                    data = baris.split()

                    if len(data) < 5:
                        continue

                    kelas = int(data[0])

                    if kelas >= len(NAMA_KELAS):
                        continue

                    x, y, w, h = map(float, data[1:5])

                    # Mengubah koordinat YOLO menjadi piksel
                    x1 = int((x - w / 2) * lebar)
                    y1 = int((y - h / 2) * tinggi)
                    x2 = int((x + w / 2) * lebar)
                    y2 = int((y + h / 2) * tinggi)

                    warna = WARNA[kelas]

                    # Menggambar bounding box sesuai kelas
                    cv2.rectangle(
                        gambar,
                        (x1, y1),
                        (x2, y2),
                        warna,
                        2
                    )

                    # Menentukan posisi dan nama kelas
                    teks = NAMA_KELAS[kelas]

                    ukuran, baseline = cv2.getTextSize(
                        teks,
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.55,
                        2
                    )

                    tx = max(0, min(x1, lebar - ukuran[0]))
                    ty = y1 - 5

                    # Menyesuaikan posisi teks agar tidak keluar gambar
                    if ty - ukuran[1] < 0:
                        ty = min(
                            y1 + ukuran[1] + 8,
                            tinggi - baseline
                        )

                    # Membuat latar belakang teks
                    cv2.rectangle(
                        gambar,
                        (tx, ty - ukuran[1] - 5),
                        (tx + ukuran[0] + 4, ty + baseline),
                        warna,
                        -1
                    )

                    # Menampilkan nama kelas pada bounding box
                    cv2.putText(
                        gambar,
                        teks,
                        (tx + 2, ty - 2),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.55,
                        (255, 255, 255),
                        2
                    )

        # Menampilkan hasil visualisasi
        ax.imshow(gambar)
        ax.set_title(os.path.basename(path), fontsize=8)
        ax.axis("off")

    plt.suptitle(f"VISUALISASI DATASET {s.upper()}", fontsize=16)
    plt.tight_layout()
    plt.show()

**LANGKAH 6: Training (100 epoch)**

1. Cek lokasi file YAML dan konfigurasi

In [ ]:
import os
import yaml

# Menentukan direktori proyek dan file konfigurasi dataset
PROJECT_DIR = "/content/drive/MyDrive/ProjekUTS_Kelompok1PCD"
DATASET_DIR = f"{PROJECT_DIR}/datasets"
DATA_YAML = f"{DATASET_DIR}/data_helm.yaml"

# Memeriksa keberadaan file YAML
print("File YAML tersedia:", os.path.exists(DATA_YAML))

# Membaca konfigurasi dataset jika file tersedia
if os.path.exists(DATA_YAML):
    with open(DATA_YAML, "r") as file:
        config = yaml.safe_load(file)

    # Menampilkan isi konfigurasi dataset
    print("\nIsi konfigurasi:")
    print(config)

2. Cek struktur folder dataset

In [ ]:
from pathlib import Path

# Menentukan pembagian dataset
splits = ["train", "valid", "test"]

# Memeriksa folder dan jumlah file pada setiap dataset
for split in splits:
    image_dir = DATASET_DIR + f"/{split}/images"
    label_dir = DATASET_DIR + f"/{split}/labels"

    print(f"\n=== {split.upper()} ===")

    # Memeriksa keberadaan folder images dan labels
    print("Folder images :", os.path.exists(image_dir))
    print("Folder labels :", os.path.exists(label_dir))

    # Menghitung jumlah gambar
    if os.path.exists(image_dir):
        print("Jumlah gambar :", len(list(Path(image_dir).glob("*"))))

    # Menghitung jumlah file anotasi
    if os.path.exists(label_dir):
        print("Jumlah label  :", len(list(Path(label_dir).glob("*.txt"))))

3. Cek kesesuaian gambar dan label

In [ ]:
from pathlib import Path

# Ekstensi file gambar yang digunakan
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Memeriksa kesesuaian gambar dan label pada setiap dataset
for split in splits:
    image_dir = Path(DATASET_DIR) / split / "images"
    label_dir = Path(DATASET_DIR) / split / "labels"

    # Mengambil seluruh file gambar berdasarkan ekstensi
    images = [
        f for f in image_dir.iterdir()
        if f.is_file() and f.suffix.lower() in IMAGE_EXT
    ]

    # Mengambil seluruh file anotasi
    labels = list(label_dir.glob("*.txt"))

    # Membandingkan nama file gambar dan label
    image_names = {f.stem for f in images}
    label_names = {f.stem for f in labels}

    # Mengidentifikasi gambar tanpa label dan label tanpa gambar
    missing_labels = image_names - label_names
    orphan_labels = label_names - image_names

    # Menampilkan hasil pemeriksaan
    print(f"\n=== {split.upper()} ===")
    print("Jumlah gambar       :", len(images))
    print("Jumlah label        :", len(labels))
    print("Gambar tanpa label  :", len(missing_labels))
    print("Label tanpa gambar  :", len(orphan_labels))

4. Cek format anotasi YOLO dan ID kelas

In [ ]:
from pathlib import Path

# Memeriksa validitas anotasi pada setiap dataset
for split in splits:
    label_dir = Path(DATASET_DIR) / split / "labels"

    invalid = []
    class_counts = {0: 0, 1: 0, 2: 0}

    # Membaca setiap file anotasi
    for file in label_dir.glob("*.txt"):
        with open(file, "r") as f:
            for line_no, line in enumerate(f, 1):
                parts = line.strip().split()

                if not parts:
                    continue

                try:
                    # Memeriksa jumlah kolom anotasi YOLO
                    if len(parts) != 5:
                        raise ValueError("Jumlah kolom bukan 5")

                    cls = int(parts[0])
                    x, y, w, h = map(float, parts[1:])

                    # Memastikan ID kelas sesuai dengan kelas dataset
                    if cls not in class_counts:
                        raise ValueError("ID kelas di luar 0-2")

                    # Memastikan nilai koordinat berada dalam rentang yang valid
                    if not (0 <= x <= 1 and 0 <= y <= 1
                            and 0 < w <= 1 and 0 < h <= 1):
                        raise ValueError("Koordinat tidak valid")

                    # Menghitung jumlah anotasi untuk setiap kelas
                    class_counts[cls] += 1

                except ValueError as e:
                    # Menyimpan informasi anotasi yang bermasalah
                    invalid.append(
                        f"{file.name}, baris {line_no}: {e}"
                    )

    # Menampilkan hasil pemeriksaan anotasi
    print(f"\n=== {split.upper()} ===")
    print("Jumlah anotasi per kelas:", class_counts)
    print("Jumlah anotasi bermasalah:", len(invalid))

    # Menampilkan maksimal 10 kesalahan pertama
    for error in invalid[:10]:
        print(error)

5. Cek GPU dan Ultralytics

In [ ]:
import torch
import ultralytics

# Menampilkan versi library yang digunakan
print("Versi Ultralytics:", ultralytics.__version__)
print("Versi PyTorch    :", torch.__version__)

# Memeriksa ketersediaan CUDA untuk penggunaan GPU
print("CUDA tersedia    :", torch.cuda.is_available())

# Menampilkan informasi GPU jika CUDA tersedia
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memori GPU:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

6. Cek model pretrained dan folder penyimpanan

In [ ]:
from ultralytics import YOLO
import os

# Memuat model YOLO26s
model = YOLO("yolo26s.pt")

# Menentukan folder penyimpanan hasil pelatihan
SAVE_DIR = f"{PROJECT_DIR}/runs"

# Membuat folder penyimpanan jika belum tersedia
os.makedirs(SAVE_DIR, exist_ok=True)

# Memastikan model berhasil dimuat dan folder tersedia
print("Model YOLO26s berhasil dimuat.")
print("Folder penyimpanan tersedia:", os.path.exists(SAVE_DIR))

7. Training Data

In [ ]:
from ultralytics import YOLO

# Fungsi utama untuk menjalankan proses training
def main():
    model = YOLO("yolo26s.pt")

    results = model.train(
        data="/content/drive/MyDrive/ProjekUTS_Kelompok1PCD/datasets/data_helm.yaml",
        epochs=100,
        patience=20,
        imgsz=640,
        batch=16,
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        close_mosaic=10,
        box=7.5,
        cls=1.5,
        device=0,
        workers=4,
        plots=True,
        project="/content/drive/MyDrive/ProjekUTS_Kelompok1PCD/runs",
        name="train-data-hasil",
        exist_ok=True
    )

# Menjalankan fungsi utama
if __name__ == "__main__":
    main()

**Lanjut Train Data (Alasannya: GPU Limit)**

In [ ]:
from google.colab import drive

# Menghubungkan Google Drive ke Google Colab
drive.mount('/content/drive')

In [ ]:
# Install library YOLO dan Roboflow
!pip install -q ultralytics roboflow

In [ ]:
from ultralytics import YOLO

# Memuat model dari checkpoint training terakhir
model = YOLO("/content/drive/MyDrive/ProjekUTS_Kelompok1PCD/runs/train-data-hasil/weights/last.pt")

# Melanjutkan training dari epoch terakhir yang tersimpan
results = model.train(resume=True)

**Lanjut Train Data (Alasannya: GPU Limit)**

In [ ]:
from google.colab import drive

# Menghubungkan Google Drive ke Google Colab
drive.mount('/content/drive')

In [ ]:
# Install library YOLO dan Roboflow
!pip install -q ultralytics roboflow

In [ ]:
from ultralytics import YOLO

# Memuat model dari checkpoint training terakhir
model = YOLO("/content/drive/MyDrive/ProjekUTS_Kelompok1PCD/runs/train-data-hasil/weights/last.pt")

# Melanjutkan training dari epoch terakhir yang tersimpan
results = model.train(resume=True)

**LANGKAH 7: Evaluasi di data test dan visualisasi hasil**

1. Siapkan path dan muat model terbaik

In [ ]:
import os
from google.colab import drive
from ultralytics import YOLO

# Sambungkan Drive (aman dijalankan ulang kalau sesi Colab baru)
drive.mount('/content/drive')

# Path proyek (sama seperti langkah sebelumnya)
PROJECT_DIR = "/content/drive/MyDrive/ProjekUTS_Kelompok1PCD"
DATASET_DIR = f"{PROJECT_DIR}/datasets"
DATA_YAML   = f"{DATASET_DIR}/data_helm.yaml"
BEST_PT     = f"{PROJECT_DIR}/runs/train-data-hasil/weights/best.pt"

# Folder untuk menyimpan gambar hasil visualisasi (untuk laporan dan slide)
OUT_VIS = f"{PROJECT_DIR}/hasil_visualisasi"
os.makedirs(OUT_VIS, exist_ok=True)

# Nama kelas dan warna kotak (RGB), konsisten dengan visualisasi dataset
NAMA_KELAS = ["helmet", "no-helmet", "pengendara-motor"]
WARNA = [(0, 200, 0), (220, 0, 0), (0, 120, 255)]

# Pakai best.pt, bukan last.pt
model = YOLO(BEST_PT)
print("Model dimuat:", os.path.exists(BEST_PT))

2. Evaluasi di data test

In [ ]:
import os

T = f"{DATASET_DIR}/test"

# Isi folder test (level pertama)
print("Isi folder test:", sorted(os.listdir(T)))
print()

# Rincian tiap isi: folder atau file, dan jumlah file di dalamnya
for nama in sorted(os.listdir(T)):
    p = f"{T}/{nama}"
    if os.path.isdir(p):
        isi = os.listdir(p)
        print(f"[FOLDER] {nama:15s} -> {len(isi)} item, contoh: {isi[:3]}")
    else:
        print(f"[FILE]   {nama}")

# Hitung file per ekstensi di seluruh isi folder test
print()
from collections import Counter
ekstensi = Counter()
for root, dirs, files in os.walk(T):
    for f in files:
        ekstensi[os.path.splitext(f)[1].lower()] += 1
print("Jumlah file per ekstensi:", dict(ekstensi))

# Apakah folder labels ada dan berapa isinya
lbl = f"{T}/labels"
print("\nFolder labels ada:", os.path.exists(lbl))
i

In [ ]:
import os
from collections import Counter
from pathlib import Path

T = Path(DATASET_DIR) / "test"
gambar = {f.stem for f in (T/"images").iterdir()}
label  = {f.stem for f in (T/"labels").glob("*.txt")}
print("Gambar tanpa label:", len(gambar - label))
print("Label tanpa gambar:", len(label - gambar))

# Isi label: kelas harus hanya 0, 1, 2 dan koordinat valid
hitung, bermasalah, kosong = Counter(), [], 0
for f in (T/"labels").glob("*.txt"):
    baris = [b.split() for b in open(f) if b.strip()]
    if not baris:
        kosong += 1
    for q in baris:
        try:
            if len(q) != 5: raise ValueError("kolom bukan 5")
            k = int(q[0]); x, y, w, h = map(float, q[1:])
            if k not in (0, 1, 2): raise ValueError("kelas di luar 0-2")
            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                raise ValueError("koordinat tidak valid")
            hitung[k] += 1
        except ValueError as e:
            bermasalah.append(f"{f.name}: {e}")
print("Objek per kelas:", dict(sorted(hitung.items())))
print("Label kosong:", kosong, "| bermasalah:", len(bermasalah))
print(bermasalah[:5])

# Pastikan tidak ada tumpang tindih dan cache lama tidak ada
for s in ["train", "valid"]:
    print(f"test ∩ {s}:", len(gambar & {f.stem for f in (Path(DATASET_DIR)/s/'images').iterdir()}))
cache = T / "labels.cache"
if cache.exists():
    cache.unlink(); print("Cache test lama dihapus")

In [ ]:
yaml_test = f"{DATASET_DIR}/data_test.yaml"
open(yaml_test, "w").write(f"""path: {DATASET_DIR}
train: test/images
val: test/images
test: test/images
nc: 3
names: ['helmet', 'no-helmet', 'pengendara-motor']
""")

metrics = model.val(data=yaml_test, imgsz=640, batch=16, plots=True,
                    project=f"{PROJECT_DIR}/runs", name="evaluasi-test", exist_ok=True)

**3. Tabel hasil per kelas (siap dimasukkan ke laporan)**

In [ ]:
import pandas as pd

# Ambil metrik per kelas dari hasil evaluasi test
baris = []
for i, k in enumerate(metrics.box.ap_class_index):
    baris.append({
        "Kelas": NAMA_KELAS[k],
        "Precision": round(float(metrics.box.p[i]), 3),
        "Recall": round(float(metrics.box.r[i]), 3),
        "mAP50": round(float(metrics.box.ap50[i]), 3),
        "mAP50-95": round(float(metrics.box.all_ap[i].mean()), 3),
    })

# Baris rata-rata semua kelas
baris.append({
    "Kelas": "SEMUA",
    "Precision": round(float(metrics.box.mp), 3),
    "Recall": round(float(metrics.box.mr), 3),
    "mAP50": round(float(metrics.box.map50), 3),
    "mAP50-95": round(float(metrics.box.map), 3),
})

tabel = pd.DataFrame(baris)
print(tabel.to_string(index=False))

# Simpan ke CSV di Drive
tabel.to_csv(f"{OUT_VIS}/hasil_evaluasi_test.csv", index=False)

**4. Confusion matrix, kurva PR dan F1, kurva training**

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Folder hasil evaluasi test dan folder hasil training
EVAL_DIR  = str(metrics.save_dir)
TRAIN_DIR = f"{PROJECT_DIR}/runs/train-data-hasil"

# Daftar plot yang ditampilkan
plot_list = [
    ("Confusion Matrix (Test)",         f"{EVAL_DIR}/confusion_matrix.png"),
    ("Confusion Matrix Ternormalisasi", f"{EVAL_DIR}/confusion_matrix_normalized.png"),
    ("Kurva Precision-Recall (Test)",   f"{EVAL_DIR}/BoxPR_curve.png"),
    ("Kurva F1 (Test)",                 f"{EVAL_DIR}/BoxF1_curve.png"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (judul, path) in zip(axes.ravel(), plot_list):
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
    else:
        ax.text(0.5, 0.5, "File tidak ditemukan", ha="center")
        print("Tidak ada:", path)
    ax.set_title(judul, fontsize=12)
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT_VIS}/evaluasi_test_plots.png", dpi=150)
plt.show()

# Kurva loss dan metrik selama training
hasil_train = f"{TRAIN_DIR}/results.png"
if os.path.exists(hasil_train):
    plt.figure(figsize=(16, 8))
    plt.imshow(mpimg.imread(hasil_train))
    plt.title("Kurva Training (loss dan metrik per epoch)")
    plt.axis("off")
    plt.show()

**5. Contoh hasil deteksi di gambar test**

In [ ]:
import glob, random, cv2
import matplotlib.pyplot as plt

def gambar_prediksi(img_rgb, hasil):
    # Gambar kotak prediksi sesuai warna kelas, lengkap dengan confidence
    img = img_rgb.copy()
    tinggi, lebar = img.shape[:2]
    for kotak, k, conf in zip(hasil.boxes.xyxyn.cpu().numpy(),
                              hasil.boxes.cls.cpu().numpy().astype(int),
                              hasil.boxes.conf.cpu().numpy()):
        x1, y1 = int(kotak[0]*lebar), int(kotak[1]*tinggi)
        x2, y2 = int(kotak[2]*lebar), int(kotak[3]*tinggi)
        cv2.rectangle(img, (x1, y1), (x2, y2), WARNA[k], 2)
        teks = f"{NAMA_KELAS[k]} {conf:.2f}"
        (tw, th), bl = cv2.getTextSize(teks, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        # Latar teks supaya mudah dibaca
        cv2.rectangle(img, (x1, max(y1 - th - 6, 0)), (x1 + tw + 4, max(y1, th + 6)), WARNA[k], -1)
        cv2.putText(img, teks, (x1 + 2, max(y1 - 4, th + 2)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return img

# Ambil 8 gambar test acak
semua_test = glob.glob(f"{DATASET_DIR}/test/images/*")
contoh = random.sample(semua_test, 8)

# conf=0.25 = hanya tampilkan deteksi dengan keyakinan minimal 25%
hasil_list = model.predict(contoh, conf=0.25, imgsz=640, verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, p, r in zip(axes.ravel(), contoh, hasil_list):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ax.imshow(gambar_prediksi(img, r))
    ax.set_title(f"{len(r.boxes)} objek terdeteksi", fontsize=10)
    ax.axis("off")
plt.suptitle("HASIL DETEKSI PADA DATA TEST", fontsize=16)
plt.tight_layout()
plt.savefig(f"{OUT_VIS}/contoh_deteksi_test.png", dpi=150)
plt.show()

**6. Kasus gagal (no-helmet terlewat)**

In [ ]:
def baca_gt(path_label):
    # Baca kotak ground truth: daftar (kelas, x1, y1, x2, y2) skala 0-1
    hasil = []
    if os.path.exists(path_label):
        for b in open(path_label):
            q = b.split()
            if len(q) == 5:
                k = int(q[0]); cx, cy, w, h = map(float, q[1:])
                hasil.append((k, cx - w/2, cy - h/2, cx + w/2, cy + h/2))
    return hasil

def iou(a, b):
    # IoU dua kotak format (x1, y1, x2, y2)
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    irisan = ix * iy
    gabung = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - irisan
    return irisan / gabung if gabung > 0 else 0

# Cari gambar test yang punya no-helmet (GT) tapi tidak terdeteksi (IoU < 0.5)
gagal = []
total_gt_no, total_terlewat = 0, 0
semua_test = sorted(glob.glob(f"{DATASET_DIR}/test/images/*"))
for i in range(0, len(semua_test), 32):
    grup = semua_test[i:i+32]
    hasil_grup = model.predict(grup, conf=0.25, imgsz=640, verbose=False)
    for p, r in zip(grup, hasil_grup):
        nama = os.path.splitext(os.path.basename(p))[0]
        gt = [g for g in baca_gt(f"{DATASET_DIR}/test/labels/{nama}.txt") if g[0] == 1]
        kotak = r.boxes.xyxyn.cpu().numpy()
        kelas = r.boxes.cls.cpu().numpy().astype(int)
        pred_no = [tuple(k) for k, c in zip(kotak, kelas) if c == 1]
        terlewat = [g[1:] for g in gt if not any(iou(g[1:], pk) >= 0.5 for pk in pred_no)]
        total_gt_no += len(gt)
        total_terlewat += len(terlewat)
        if terlewat:
            gagal.append((p, r, terlewat))

print(f"No-helmet terlewat: {total_terlewat} dari {total_gt_no} objek")
print(f"Gambar terdampak  : {len(gagal)} dari {len(semua_test)}")

# Tampilkan maksimal 8 contoh: kotak kuning = no-helmet yang terlewat
tampil = random.sample(gagal, min(8, len(gagal)))
if tampil:
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    for ax, (p, r, terlewat) in zip(axes.ravel(), tampil):
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        img = gambar_prediksi(img, r)
        h, w = img.shape[:2]
        for x1, y1, x2, y2 in terlewat:
            cv2.rectangle(img, (int(x1*w), int(y1*h)), (int(x2*w), int(y2*h)), (255, 200, 0), 3)
        ax.imshow(img)
        ax.set_title(f"{len(terlewat)} no-helmet terlewat (kotak kuning)", fontsize=10)
        ax.axis("off")
    for ax in axes.ravel()[len(tampil):]:
        ax.axis("off")
    plt.suptitle("KASUS GAGAL: NO-HELMET TIDAK TERDETEKSI", fontsize=16)
    plt.tight_layout()
    plt.savefig(f"{OUT_VIS}/kasus_gagal_no_helmet.png", dpi=150)
    plt.show()

**LANGKAH 8: Demo aplikasi web (Gradio)**

**1. Install Gradio dan muat model**

In [ ]:
!pip install -q gradio

import os, cv2, subprocess, tempfile
import numpy as np
import gradio as gr
from ultralytics import YOLO

# Path proyek (sama seperti langkah sebelumnya)
PROJECT_DIR = "/content/drive/MyDrive/ProjekUTS_Kelompok1PCD"
BEST_PT = f"{PROJECT_DIR}/runs/train-data-hasil/weights/best.pt"
OUT_VIS = f"{PROJECT_DIR}/hasil_visualisasi"
os.makedirs(OUT_VIS, exist_ok=True)

# Nama kelas dan warna kotak (RGB), konsisten dengan visualisasi sebelumnya
NAMA_KELAS = ["helmet", "no-helmet", "pengendara-motor"]
WARNA = [(0, 200, 0), (220, 0, 0), (0, 120, 255)]

model = YOLO(BEST_PT)
print("Model dimuat:", os.path.exists(BEST_PT))

**2. Fungsi deteksi gambar dan video**

In [ ]:
def gambar_kotak(img_rgb, hasil):
    # Gambar kotak prediksi sesuai warna kelas, lengkap dengan confidence
    img = img_rgb.copy()
    for kotak, k, conf in zip(hasil.boxes.xyxy.cpu().numpy().astype(int),
                              hasil.boxes.cls.cpu().numpy().astype(int),
                              hasil.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = kotak
        cv2.rectangle(img, (x1, y1), (x2, y2), WARNA[k], 2)
        teks = f"{NAMA_KELAS[k]} {conf:.2f}"
        (tw, th), bl = cv2.getTextSize(teks, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        # Latar teks supaya mudah dibaca
        cv2.rectangle(img, (x1, max(y1 - th - 6, 0)), (x1 + tw + 4, max(y1, th + 6)), WARNA[k], -1)
        cv2.putText(img, teks, (x1 + 2, max(y1 - 4, th + 2)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return img

def ringkasan(hasil):
    # Hitung jumlah objek per kelas dan beri status pelanggaran
    kelas = hasil.boxes.cls.cpu().numpy().astype(int)
    n_helm, n_no, n_rider = (int((kelas == i).sum()) for i in range(3))
    status = "PERINGATAN: terdeteksi pengendara tanpa helm" if n_no > 0 else "Aman: tidak ada pengendara tanpa helm"
    return n_helm, n_no, n_rider, status

def deteksi_gambar(gambar, conf):
    # gambar dari Gradio berupa array RGB
    if gambar is None:
        return None, "Upload gambar dulu."
    hasil = model.predict(gambar, conf=conf, imgsz=640, verbose=False)[0]
    n_helm, n_no, n_rider, status = ringkasan(hasil)
    teks = (f"{status}\n"
            f"Helmet: {n_helm}\nNo-helmet: {n_no}\nPengendara-motor: {n_rider}")
    return gambar_kotak(gambar, hasil), teks

def deteksi_video(path_video, conf):
    # Proses video frame demi frame, maksimal 20 detik supaya cepat
    if path_video is None:
        return None, "Upload video dulu."
    cap = cv2.VideoCapture(path_video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    maks_frame = int(fps * 20)

    tmp_raw = tempfile.mktemp(suffix=".mp4")
    tmp_out = tempfile.mktemp(suffix=".mp4")
    tulis = cv2.VideoWriter(tmp_raw, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    n_frame, total_no, frame_no = 0, 0, 0
    while n_frame < maks_frame:
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        hasil = model.predict(rgb, conf=conf, imgsz=640, verbose=False)[0]
        _, n_no, _, _ = ringkasan(hasil)
        total_no += n_no
        frame_no += int(n_no > 0)
        tulis.write(cv2.cvtColor(gambar_kotak(rgb, hasil), cv2.COLOR_RGB2BGR))
        n_frame += 1
    cap.release()
    tulis.release()

    # Ubah ke H.264 supaya bisa diputar di browser
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp_raw,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", tmp_out])
    teks = (f"Frame diproses: {n_frame}\n"
            f"Frame dengan no-helmet: {frame_no}\n"
            f"Total deteksi no-helmet: {total_no}")
    return tmp_out, teks

**3. Jalankan antarmuka web**

In [ ]:
# Tab 1: upload foto, Tab 2: upload video
with gr.Blocks(title="Deteksi Helm Pengendara Motor") as demo:
    gr.Markdown("# Deteksi Helm pada Pengendara Sepeda Motor (YOLO26)\n"
                "Upload foto atau video, sistem menandai helmet (hijau), no-helmet (merah), "
                "dan pengendara-motor (biru).")

    slider = gr.Slider(0.1, 0.9, value=0.25, step=0.05, label="Ambang confidence")

    with gr.Tab("Foto"):
        in_img = gr.Image(type="numpy", label="Upload foto")
        btn_img = gr.Button("Deteksi")
        out_img = gr.Image(label="Hasil deteksi")
        out_txt = gr.Textbox(label="Ringkasan", lines=4)

        # Tombol download hasil deteksi foto
        download_img = gr.DownloadButton(label="Download Hasil Foto")

        btn_img.click(
            deteksi_gambar,
            [in_img, slider],
            [out_img, out_txt]
        )

    with gr.Tab("Video"):
        in_vid = gr.Video(label="Upload video (maks. 20 detik diproses)")
        btn_vid = gr.Button("Deteksi")
        out_vid = gr.Video(label="Hasil deteksi")
        out_txt2 = gr.Textbox(label="Ringkasan", lines=3)

        # Tombol download hasil deteksi video
        download_vid = gr.DownloadButton(label="Download Hasil Video")

        btn_vid.click(
            deteksi_video,
            [in_vid, slider],
            [out_vid, out_txt2]
        )

# share=True membuat link publik sementara yang bisa dibuka saat presentasi online
demo.launch(share=True, debug=True)

In [ ]:
# Tab 1: upload foto, Tab 2: upload video
with gr.Blocks(title="Deteksi Helm Pengendara Motor") as demo:
    gr.Markdown("# Deteksi Helm pada Pengendara Sepeda Motor (YOLO26)\n"
                "Upload foto atau video, sistem menandai helmet (hijau), no-helmet (merah), "
                "dan pengendara-motor (biru).")

    slider = gr.Slider(0.1, 0.9, value=0.25, step=0.05, label="Ambang confidence")

    with gr.Tab("Foto"):
        in_img = gr.Image(type="numpy", label="Upload foto")
        btn_img = gr.Button("Deteksi")
        out_img = gr.Image(label="Hasil deteksi")
        out_txt = gr.Textbox(label="Ringkasan", lines=4)

        btn_img.click(
            fn=deteksi_gambar,
            inputs=[in_img, slider],
            outputs=[out_img, out_txt]
        )

    with gr.Tab("Video"):
        in_vid = gr.Video(label="Upload video (maks. 20 detik diproses)")
        btn_vid = gr.Button("Deteksi")
        out_vid = gr.Video(label="Hasil deteksi")
        out_txt2 = gr.Textbox(label="Ringkasan", lines=3)

        btn_vid.click(
            fn=deteksi_video,
            inputs=[in_vid, slider],
            outputs=[out_vid, out_txt2]
        )

# Menjalankan aplikasi Gradio
demo.launch(share=True, debug=True)